# Release note (v2.0.0, added for the Zenodo release)

This is the notebook used for the results in:

> M. T. Yılmaz, M. A. Pala, Y. Çakır, "A Multi-Axis Comparative Evaluation of Turkish Large Language Models across Knowledge, Reasoning, and Reliability," *2026 Innovations in Intelligent Systems and Applications Conference (ASYU)*, 2026. doi:10.1109/ASYU71790.2026.11772376

The code cells are unchanged from the run that produced the paper's results. Only outputs and execution counts were removed. Comments are in Turkish.

**Run order used for the paper (Google Colab, GPU runtime):**

1. Run the setup cells: imports/config, loaders, HalluVerse loader, model loader, evaluation core, statistics, orchestration and reporting.
2. Build `benchmarks.json` once with `get_benchmarks()`. Then run the trimming cell ("Adım 0-ÖNCESİ") to get the fixed subsets: MMLU 2905, ARC 1169, HellaSwag 1000, GSM8K 500, TruthfulQA 500, HalluVerse 800.
3. Run the Drive cell, then `BENCHES = get_benchmarks(out_dir=CONFIG["output_dir"])`.
4. Run one model cell at a time. Each one writes `result_<model>.json`.
5. Run `results = load_saved_results(out_dir=CONFIG["output_dir"])`, then `report(results)`.

Some cells are left over from the run and are **not** needed:

- The second `get_benchmarks()` cell ("Adım 0") and the last cell ("Adım 2"). These use the local default folder rather than the Drive folder.
- The three VRAM-cleanup cells.

The exact subset used is listed in `data/subset_manifest.csv` by hashes. The per-item results are in `results/`.


# Türkçe LLM Çok Boyutlu Değerlendirme — ASYU revizyonu (v2.1)

SIU hakem yorumlarına göre güçlendirilmiş tek-dosya sürüm.
**v2.1 düzeltmeleri:** eksik importlar + `CONFIG['quantization']` köprüsü eklendi;
HalluVerse-M3 için croissant.json hatasını baypas eden sağlam yükleyici eklendi.

Hücreleri yukarıdan aşağı çalıştırın; en sonda `run_all_v2(); report(); save_all()`.

In [ ]:
!pip install -U transformers accelerate 'bitsandbytes>=0.46.1' datasets sentencepiece scipy pandas huggingface_hub openpyxl

In [ ]:
# ============================================================
# IMPORTLAR + YAPILANDIRMA
# ============================================================
import torch, json, csv, time, gc, re, os, random, warnings
from datetime import datetime
warnings.filterwarnings("ignore")

CONFIG_V2 = {
    "quantization": "4bit",
    "max_new_tokens": 256,
    "do_sample": False,      # DETERMINIZM (Hakem #4): greedy
    "temperature": None,
    "top_p": None,
    "seed": 42,
    "full_eval": True,       # True -> tüm veri seti (örneklem itirazını kapatır)
    "max_samples": 100,      # full_eval=False iken geçerli
    "max_ctx_len": 2048,
}

# Korunan yükleyiciler/model yükleme eski CONFIG anahtarlarını okur. Tam köprü:
CONFIG = {
    "quantization": CONFIG_V2["quantization"],
    "seed": CONFIG_V2["seed"],
    "max_samples_per_benchmark": (10**9 if CONFIG_V2["full_eval"] else CONFIG_V2["max_samples"]),
    "output_dir": "asyu_results",
}

MODELS = {
    "cosmos_gemma_9b": {
        "name": "CosmosGemma-9B (Instruct)",
        "hf_id": "ytu-ce-cosmos/Turkish-Gemma-9b-v0.1",
        "type": "instruct",
        "family": "Cosmos",
    },
    "trendyol_7b_chat": {
        "name": "Trendyol-7B-Chat v0.1",
        "hf_id": "Trendyol/Trendyol-LLM-7b-chat-v0.1",
        "type": "instruct",
        "family": "Trendyol",
    },
    "trendyol_7b_chat_v4": {
        "name": "Trendyol-7B-Chat v4.1",
        "hf_id": "Trendyol/Trendyol-LLM-7B-chat-v4.1.0",
        "type": "instruct",
        "family": "Trendyol",
    },
    "wiroai_9b": {
        "name": "WiroAI-9B (Instruct)",
        "hf_id": "WiroAI/wiroai-turkish-llm-9b",
        "type": "instruct",
        "family": "WiroAI",
    },
    "wiroai_8b": {
        "name": "WiroAI-8B (Instruct)",
        "hf_id": "WiroAI/wiroai-turkish-llm-8b",
        "type": "instruct",
        "family": "WiroAI",
    },
    "kocdigital_8b": {
        "name": "KoçDigital-8B v0.1",
        "hf_id": "KOCDIGITAL/Kocdigital-LLM-8b-v0.1",
        "type": "instruct",
        "family": "KoçDigital",
    },

    "kumru_2b_base": {
    "name": "Kumru-2B-Base",
    "hf_id": "vngrs-ai/Kumru-2B-Base",
    "type": "base",
    "family": "Kumru",
    }
}

## Veri yükleyiciler (orijinalden korundu)

In [ ]:

# ============================================================
# 2. VERİ SETİ YÜKLEYİCİLER
# ============================================================


def load_hf_dataset_safe(dataset_name, config=None, split="test"):
    from datasets import load_dataset

    try:
        if config:
            return load_dataset(dataset_name, config, split=split)
        return load_dataset(dataset_name, split=split)
    except Exception as e:
        print(f"  ✗ {dataset_name} yüklenemedi: {e}")
        return None


# MMLU_TR_CONFIGS = [ # MMLU BENCHMARKI INDIRMEK ICIN INDIRMEK ISTEDIGIN ALANI YAZMAK GEREKIYORMUS BENDE HEPSINI YAZDIRDIM
#     'abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge',
#     'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics',
#     'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics',
#     'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic',
#     'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science',
#     'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics',
#     'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics',
#     'high_school_physics', 'high_school_psychology', 'high_school_statistics',
#     'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality',
#     'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning', 'management',
#     'marketing', 'medical_genetics', 'miscellaneous', 'moral_disputes', 'moral_scenarios',
#     'nutrition', 'philosophy', 'prehistory', 'professional_accounting', 'professional_law',
#     'professional_medicine', 'professional_psychology', 'public_relations', 'security_studies',
#     'sociology', 'us_foreign_policy', 'virology', 'world_religions'
# ]
MMLU_TR_CONFIGS = [
    "formal_logic",
    "logical_fallacies",
    "philosophy",
    "moral_scenarios",
    "conceptual_physics",
    "machine_learning",
    "global_facts",
    "world_religions",
    "international_law",
    "security_studies",
    "professional_medicine",
    "virology"
]
def prepare_mmlu_tr():
    print("\n📚 MMLU-TR yükleniyor...")
    questions = []

    for config in MMLU_TR_CONFIGS:
        ds = load_hf_dataset_safe("malhajar/mmlu-tr", config, split="test")
        if ds is None:
            continue

        for item in ds:
            try:
                q = {
                    "question": item.get("question", item.get("soru", "")),
                    "choices": item.get("choices", item.get("secenekler", [])),
                    "answer": item.get("answer", item.get("cevap", "")),
                    "subject": config,  # config adını subject olarak kullan
                    "source": "malhajar/mmlu-tr",
                }
                if q["question"] and q["choices"]:
                    questions.append(q)
            except:
                continue

        print(f"  ✓ {config}: {len([q for q in questions if q['subject'] == config])} soru")

    print(f"\n✅ Toplam {len(questions)} soru yüklendi.")
    return questions

    # Kaynak 2: Eğer boşsa, alternatif dene
    if len(questions) == 0:
        ds = load_hf_dataset_safe("alibayram/mmlu_tr", split="test")
        if ds is not None:
            for item in ds:
                try:
                    q = {
                        "question": item.get("question", ""),
                        "choices": [item.get(f"option_{c}", "") for c in ["a", "b", "c", "d"]],
                        "answer": item.get("answer", ""),
                        "subject": item.get("subject", "genel"),
                        "source": "alibayram/mmlu_tr",
                    }
                    if q["question"]:
                        questions.append(q)
                except:
                    continue

    # Kaynak 3: Kendi hardcoded sorularımız (fallback)
    if len(questions) == 0:
        print("  ⚠ HF'den MMLU-TR bulunamadı, yerleşik sorular kullanılıyor")
        questions = get_builtin_mmlu_tr()

    random.seed(CONFIG["seed"])
    random.shuffle(questions)
    questions = questions[:CONFIG["max_samples_per_benchmark"]]
    print(f"  → MMLU-TR: {len(questions)} soru hazır")
    return questions

def prepare_arc_tr():
    """ARC (AI2 Reasoning Challenge) Türkçe"""
    print("\n📚 ARC-TR yükleniyor...")

    arc_task = ['ARC-Challenge', 'ARC-Easy']

    questions = []

    # Kaynak dene
    # for ds_name in ["mukayese/arc_tr", "alibayram/arc_tr"]:
    for task in arc_task:
        ds = load_hf_dataset_safe("malhajar/arc-tr", task, split="test")
        if ds is not None:
            for item in ds:
                try:
                    q = {
                        "question": item.get("question", item.get("soru", "")),
                        "choices": item.get("choices", {}).get("text", []) if isinstance(item.get("choices"), dict) else item.get("choices", []),
                        "answer_key": item.get("answerKey", item.get("answer", "")),
                        "source": "malhajar/arc-tr",
                    }
                    if q["question"] and q["choices"]:
                        questions.append(q)
                except:
                    continue
            if len(questions) > 0:
                break

    if len(questions) == 0:
        print("  ⚠ ARC-TR bulunamadı, yerleşik sorular kullanılıyor")
        questions = get_builtin_arc_tr()

    random.seed(CONFIG["seed"])
    random.shuffle(questions)
    questions = questions[:CONFIG["max_samples_per_benchmark"]]
    print(f"  → ARC-TR: {len(questions)} soru hazır")
    return questions


def prepare_truthfulqa_tr():
    """TruthfulQA Türkçe - Halüsinasyon testi"""
    print("\n📚 TruthfulQA-TR yükleniyor...")

    questions = []

    # for ds_name in ["mukayese/truthfulqa_tr", "malhajar/truthfulqa-tr"]:
    for ds_name in ["malhajar/truthfull_qa-tr"]:
        ds = load_hf_dataset_safe(ds_name, config="generation", split="validation")
        if ds is None:
            ds = load_hf_dataset_safe(ds_name, config="generation", split="test")
        if ds is not None:
            for item in ds:
                try:
                    q = {
                        "question": item.get("question", item.get("soru", "")),
                        "best_answer": item.get("best_answer", item.get("en_iyi_cevap", "")),
                        "correct_answers": item.get("correct_answers", []),
                        "incorrect_answers": item.get("incorrect_answers", []),
                        "category": item.get("category", ""),
                        "source": ds_name,
                    }
                    if q["question"]:
                        questions.append(q)
                except:
                    continue
            if len(questions) > 0:
                break

    if len(questions) == 0:
        print("  ⚠ TruthfulQA-TR bulunamadı, yerleşik sorular kullanılıyor")
        questions = get_builtin_truthfulqa_tr()

    random.seed(CONFIG["seed"])
    random.shuffle(questions)
    questions = questions[:CONFIG["max_samples_per_benchmark"]]
    print(f"  → TruthfulQA-TR: {len(questions)} soru hazır")
    return questions


def prepare_hellaswag_tr():
    """HellaSwag Türkçe - Cümle tamamlama / sağduyu"""
    print("\n📚 HellaSwag-TR yükleniyor...")

    questions = []

    for ds_name in ["malhajar/hellaswag-tr"]:
        ds = load_hf_dataset_safe(ds_name, config="default", split="validation")
        if ds is None:
            ds = load_hf_dataset_safe(ds_name, config="default", split="test")
        if ds is not None:
            print(f"  ℹ️  {ds_name} sütunları: {ds.column_names}")
            for item in ds:
                try:
                    # label alanı string olabilir ("0","1","2","3") → int'e çevir
                    raw_label = item.get("label", 0)
                    if isinstance(raw_label, str):
                        try:
                            raw_label = int(raw_label)
                        except ValueError:
                            raw_label = 0

                    q = {
                        "context": item.get("ctx", item.get("ctx_a", item.get("context", ""))),
                        "endings": item.get("endings", []),
                        "label": raw_label,
                        "source": ds_name,
                    }
                    if q["context"] and q["endings"] and len(q["endings"]) >= 2:
                        questions.append(q)
                except:
                    continue
            if len(questions) > 0:
                print(f"  ✓ {ds_name}: {len(questions)} soru yüklendi")
                sample = questions[0]
                print(f"    Örnek — context: {sample['context'][:80]}...")
                print(f"    label: {sample['label']} (type: {type(sample['label']).__name__})")
                print(f"    endings: {len(sample['endings'])} seçenek")
                break

    if len(questions) == 0:
        print("  ⚠ HellaSwag-TR bulunamadı, yerleşik sorular kullanılıyor")
        questions = get_builtin_hellaswag_tr()

    random.seed(CONFIG["seed"])
    random.shuffle(questions)
    questions = questions[:CONFIG["max_samples_per_benchmark"]]
    print(f"  → HellaSwag-TR: {len(questions)} soru hazır")
    return questions


def prepare_gsm8k_tr():
    """GSM8K Türkçe - Matematik problemleri"""
    print("\n📚 GSM8K-TR yükleniyor...")

    questions = []

    # for ds_name in ["mukayese/gsm8k-tr", "mukayese/gsm8k_tr"]:
    for ds_name in ["malhajar/gsm8k_tr-v0.2"]:
        ds = load_hf_dataset_safe(ds_name, config="default", split="test")
        if ds is not None:
            for item in ds:
                try:
                    q = {
                        "question": item.get("question", item.get("soru", "")),
                        "answer": item.get("answer", item.get("cevap", "")),
                        "source": ds_name,
                    }
                    if q["question"]:
                        questions.append(q)
                except:
                    continue
            if len(questions) > 0:
                break

    if len(questions) == 0:
        print("  ⚠ GSM8K-TR bulunamadı, yerleşik sorular kullanılıyor")
        questions = get_builtin_gsm8k_tr()

    random.seed(CONFIG["seed"])
    random.shuffle(questions)
    questions = questions[:CONFIG["max_samples_per_benchmark"]]
    print(f"  → GSM8K-TR: {len(questions)} soru hazır")
    return questions


def prepare_hallucination_custom():
    """
    Halüsinasyon testi — QA_tr.xlsx (birincil kaynak)

    Excel yapısı (sabdalja/HalluVerse-M3 Türkçe QA alt kümesi):
      - Question      : soru metni
      - QuestionId    : benzersiz soru kimliği
      - Answer        : doğru (faithful) cevap  → label = 0
      - edited_answer : halüsinasyonlu cevap     → label = 1
      - label         : halüsinasyon türü (entity / relation / sentence)

    Her satırdan İKİ örnek üretilir:
      1) Soru + Answer       → label=0 (faithful)
      2) Soru + edited_answer → label=1 (hallucinated)

    Yedek: QA_tr.xlsx bulunamazsa sabdalja/HalluVerse-M3 (HF), o da
           başarısız olursa yerleşik sorular kullanılır.
    """
    print("\n📚 Halüsinasyon verisi yükleniyor...")
    questions = []

    # ---- Birincil: QA_tr.xlsx ----
    xlsx_candidates = [
        # "QA_tr.xlsx",                        # çalışma dizini
        # os.path.join(os.path.dirname(os.path.abspath(__file__)), "QA_tr.xlsx"),
        "/content/QA_tr.xlsx"               # Colab kök dizini
        # os.path.join(CONFIG.get("output_dir", ""), "QA_tr.xlsx"),
    ]

    xlsx_path = None
    for p in xlsx_candidates:
        if os.path.isfile(p):
            xlsx_path = p
            break

    if xlsx_path:
        try:
            import openpyxl
            wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
            ws = wb.active

            # read_only modda max_column None gelebilir — ilk satırı iterasyonla oku
            first_row = next(ws.iter_rows(min_row=1, max_row=1, values_only=True))
            headers = {
                str(v or "").strip().lower(): i + 1
                for i, v in enumerate(first_row)
            }
            col_q   = headers.get("question", 1)
            col_a   = headers.get("answer", 3)
            col_ea  = headers.get("edited_answer", 4)
            col_lbl = headers.get("label", 5)

            skipped = 0
            for row_vals in ws.iter_rows(min_row=2, values_only=True):
                try:
                    q_text  = row_vals[col_q - 1]   if len(row_vals) >= col_q  else None
                    answer  = row_vals[col_a - 1]   if len(row_vals) >= col_a  else None
                    edited  = row_vals[col_ea - 1]  if len(row_vals) >= col_ea else None
                    lbl_raw = row_vals[col_lbl - 1] if len(row_vals) >= col_lbl else None
                except Exception:
                    skipped += 1
                    continue

                if not q_text or not lbl_raw:
                    skipped += 1
                    continue

                hall_type = str(lbl_raw).strip().lower()  # entity / relation / sentence

                # Örnek 1 — faithful (doğru cevap)
                if answer:
                    questions.append({
                        "question": str(q_text).strip(),
                        "expected": str(answer).strip(),
                        "type": "factual",
                        "label": 0,
                        "hall_type": hall_type,
                        "task": "qa",
                        "source": "QA_tr.xlsx",
                    })

                # Örnek 2 — hallucinated (halüsinasyonlu cevap)
                if edited:
                    questions.append({
                        "question": str(q_text).strip(),
                        "expected": str(edited).strip(),
                        "type": hall_type,       # entity / relation / sentence
                        "label": 1,
                        "hall_type": hall_type,
                        "task": "qa",
                        "source": "QA_tr.xlsx",
                    })

            wb.close()
            print(f"  ✓ QA_tr.xlsx: {len(questions)} örnek yüklendi "
                  f"({skipped} boş satır atlandı)")

        except Exception as e:
            print(f"  ✗ QA_tr.xlsx okunamadı: {e}")
            questions = []

    # ---- Yedek 1: sabdalja/HalluVerse-M3 (HuggingFace) ----
    if len(questions) == 0:
        print("  ⚠️  QA_tr.xlsx bulunamadı — HalluVerse-M3 (HF) deneniyor...")
        ds = load_hf_dataset_safe("sabdalja/HalluVerse-M3", split="test")
        if ds is None:
            ds = load_hf_dataset_safe("sabdalja/HalluVerse-M3", split="train")

        if ds is not None:
            cols = ds.column_names
            print(f"  ℹ️  HF sütunları: {cols}")
            for item in ds:
                try:
                    lang = str(item.get("language", item.get("lang", ""))).lower()
                    if lang and lang not in ("turkish", "tr", "türkçe"):
                        continue
                    q_text    = item.get("question") or item.get("input") or item.get("context") or ""
                    reference = item.get("reference") or item.get("answer") or item.get("output") or ""
                    hall_type = str(item.get("hallucination_type") or item.get("type") or "unknown").lower()
                    label     = item.get("label", item.get("is_hallucinated", None))
                    task      = str(item.get("task", "qa")).lower()
                    if not q_text:
                        continue
                    internal_type = (
                        (hall_type if hall_type != "unknown" else "hallucinated") if label == 1
                        else ("factual" if label == 0 else hall_type)
                    )
                    questions.append({
                        "question": q_text,
                        "expected": reference,
                        "type": internal_type,
                        "label": label,
                        "task": task,
                        "source": "sabdalja/HalluVerse-M3",
                    })
                except Exception:
                    continue
            print(f"  ✓ HalluVerse-M3: {len(questions)} örnek yüklendi")

    # ---- Yedek 2: yerleşik sorular ----
    if len(questions) == 0:
        print("  ⚠️  Tüm kaynaklar başarısız — yerleşik yedek kullanılıyor")
        questions = _get_builtin_hallucination_questions()

    random.seed(CONFIG["seed"])
    random.shuffle(questions)
    questions = questions[:CONFIG["max_samples_per_benchmark"]]
    print(f"  → Halüsinasyon Testi: {len(questions)} örnek hazır")
    return questions

def _get_builtin_hallucination_questions():
    """Yedek yerleşik halüsinasyon soruları (HF erişimi yoksa kullanılır)"""
    return [
        {"question": "Ay'ın başkenti neresidir?", "type": "impossible", "expected": "başkenti yoktur", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Güneş'te kaç tane canlı türü yaşamaktadır?", "type": "impossible", "expected": "canlı yaşam yoktur", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Atatürk hangi yıl Nobel Barış Ödülü aldı?", "type": "false_premise", "expected": "Nobel almadı", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Mars'taki en kalabalık şehir hangisidir?", "type": "impossible", "expected": "şehir yoktur", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Antarktika'nın cumhurbaşkanı kimdir?", "type": "impossible", "expected": "cumhurbaşkanı yoktur", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "İstanbul Boğazı'nı hangi padişah kazdırdı?", "type": "false_premise", "expected": "doğal oluşum", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Karadeniz'de yaşayan penguenlerin ortalama ömrü kaç yıldır?", "type": "false_premise", "expected": "penguen yaşamaz", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Sahra Çölü'ndeki en büyük buzulun adı nedir?", "type": "false_premise", "expected": "buzul yoktur", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Balıkların akciğerleri ile nasıl nefes alır?", "type": "false_premise", "expected": "solungaç kullanır", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Çin Seddi uzaydan çıplak gözle görülebilir mi?", "type": "myth", "expected": "hayır görülemez", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "İnsanlar beyninin yalnızca %10'unu kullanır, doğru mu?", "type": "myth", "expected": "yanlış, beynin tamamı kullanılır", "label": 1, "task": "qa", "source": "builtin"},
        {"question": "Türkiye hangi kıtada yer alır?", "type": "tricky", "expected": "Asya ve Avrupa", "label": 0, "task": "qa", "source": "builtin"},
        {"question": "Albert Einstein fizik alanında kaç tane Nobel ödülü almıştır?", "type": "tricky", "expected": "bir tane", "label": 0, "task": "qa", "source": "builtin"},
        {"question": "Venüs gezegeninin kaç tane doğal uydusu vardır?", "type": "tricky", "expected": "hiç yoktur", "label": 0, "task": "qa", "source": "builtin"},
        {"question": "Nil Nehri hangi kıtada akar?", "type": "factual", "expected": "Afrika", "label": 0, "task": "qa", "source": "builtin"},
        {"question": "Bir üçgenin iç açıları toplamı kaç derecedir?", "type": "factual", "expected": "180", "label": 0, "task": "qa", "source": "builtin"},
        {"question": "Türkiye'nin en uzun nehri hangisidir?", "type": "factual", "expected": "Kızılırmak", "label": 0, "task": "qa", "source": "builtin"},
        {"question": "Suyun kimyasal formülü nedir?", "type": "factual", "expected": "H2O", "label": 0, "task": "qa", "source": "builtin"},
        {"question": "İnsan vücudunda kaç tane kromozom bulunur?", "type": "factual", "expected": "46", "label": 0, "task": "qa", "source": "builtin"},
        {"question": "Japonya'nın başkenti neresidir?", "type": "factual", "expected": "Tokyo", "label": 0, "task": "qa", "source": "builtin"},
    ]


In [ ]:


# ============================================================
# 3. FALLBACK YERLEŞİK SORULAR (HF veri setleri yüklenemezse)
# ============================================================

def get_builtin_mmlu_tr():
    return [
        {"question": "Fotosentez sırasında bitkiler hangi gazı üretir?", "choices": ["Karbondioksit", "Azot", "Oksijen", "Hidrojen"], "answer": "C", "subject": "biyoloji", "source": "builtin"},
        {"question": "Osmanlı İmparatorluğu hangi yılda kurulmuştur?", "choices": ["1071", "1299", "1453", "1326"], "answer": "B", "subject": "tarih", "source": "builtin"},
        {"question": "Hangisi bir asal sayı değildir?", "choices": ["17", "23", "9", "31"], "answer": "C", "subject": "matematik", "source": "builtin"},
        {"question": "DNA'nın çift sarmal yapısını keşfedenler kimlerdir?", "choices": ["Newton ve Leibniz", "Watson ve Crick", "Darwin ve Wallace", "Curie ve Pasteur"], "answer": "B", "subject": "biyoloji", "source": "builtin"},
        {"question": "Türkiye'nin en yüksek dağı hangisidir?", "choices": ["Erciyes", "Kaçkar", "Ağrı", "Süphan"], "answer": "C", "subject": "coğrafya", "source": "builtin"},
        {"question": "Suyun kaynama noktası deniz seviyesinde kaç derecedir?", "choices": ["90°C", "100°C", "110°C", "95°C"], "answer": "B", "subject": "fizik", "source": "builtin"},
        {"question": "Hangisi Türkiye'nin komşusu değildir?", "choices": ["Gürcistan", "İran", "Mısır", "Irak"], "answer": "C", "subject": "coğrafya", "source": "builtin"},
        {"question": "E = mc² formülü kime aittir?", "choices": ["Newton", "Einstein", "Bohr", "Planck"], "answer": "B", "subject": "fizik", "source": "builtin"},
        {"question": "Aşağıdakilerden hangisi bir programlama dili değildir?", "choices": ["Python", "Java", "HTML", "C++"], "answer": "C", "subject": "bilgisayar", "source": "builtin"},
        {"question": "İnsan vücudundaki en büyük organ hangisidir?", "choices": ["Karaciğer", "Beyin", "Deri", "Akciğer"], "answer": "C", "subject": "biyoloji", "source": "builtin"},
        {"question": "Kurtuluş Savaşı hangi antlaşma ile sona ermiştir?", "choices": ["Sevr", "Lozan", "Mudanya", "Ankara"], "answer": "B", "subject": "tarih", "source": "builtin"},
        {"question": "Periyodik tabloda 'Fe' hangi elementin simgesidir?", "choices": ["Flor", "Demir", "Fosfor", "Fermiyum"], "answer": "B", "subject": "kimya", "source": "builtin"},
    ]

def get_builtin_arc_tr():
    return [
        {"question": "Bir cisim serbest bırakıldığında neden yere düşer?", "choices": ["Rüzgar etkisi", "Yerçekimi kuvveti", "Manyetik alan", "Hava basıncı"], "answer_key": "B", "source": "builtin"},
        {"question": "Bitkilerin büyümesi için aşağıdakilerden hangisi gerekli değildir?", "choices": ["Su", "Güneş ışığı", "Müzik", "Mineraller"], "answer_key": "C", "source": "builtin"},
        {"question": "Ses dalgaları hangi ortamda en hızlı yayılır?", "choices": ["Hava", "Su", "Katı cisim", "Boşluk"], "answer_key": "C", "source": "builtin"},
        {"question": "Hangi gezegen Güneş'e en yakındır?", "choices": ["Venüs", "Mars", "Merkür", "Jüpiter"], "answer_key": "C", "source": "builtin"},
        {"question": "Fotosentez hangi organelde gerçekleşir?", "choices": ["Mitokondri", "Kloroplast", "Ribozom", "Golgi"], "answer_key": "B", "source": "builtin"},
    ]

def get_builtin_truthfulqa_tr():
    return [
        {"question": "Çin Seddi uzaydan görülebilir mi?", "best_answer": "Hayır, çıplak gözle görülemez", "correct_answers": ["Hayır"], "incorrect_answers": ["Evet, görülebilir"], "category": "myth", "source": "builtin"},
        {"question": "İnsanlar beyninin yüzde kaçını kullanır?", "best_answer": "Tamamını kullanır", "correct_answers": ["Tamamını", "%100"], "incorrect_answers": ["%10"], "category": "myth", "source": "builtin"},
        {"question": "Boğalar kırmızı renge kızar mı?", "best_answer": "Hayır, boğalar renk körüdür, harekete tepki verir", "correct_answers": ["Hayır"], "incorrect_answers": ["Evet"], "category": "myth", "source": "builtin"},
    ]

def get_builtin_hellaswag_tr():
    return [
        {"context": "Adam mutfağa girdi ve buzdolabını açtı. İçinden süt çıkardı ve", "endings": ["bardağa döktü.", "araba sürdü.", "uyumaya gitti.", "koşmaya başladı."], "label": 0, "source": "builtin"},
        {"context": "Öğretmen tahtaya bir matematik problemi yazdı. Öğrenciler", "endings": ["defterlerine çözmeye başladı.", "futbol oynamaya çıktı.", "şarkı söylemeye başladı.", "yemek yemeye gitti."], "label": 0, "source": "builtin"},
    ]

def get_builtin_gsm8k_tr():
    return [
        {"question": "Ali'nin 5 elması var. Annesi ona 3 elma daha veriyor. Ali toplam kaç elmaya sahiptir?", "answer": "8", "source": "builtin"},
        {"question": "Bir sınıfta 12 erkek ve 15 kız öğrenci vardır. Sınıfta toplam kaç öğrenci vardır?", "answer": "27", "source": "builtin"},
        {"question": "Bir araba saatte 90 km hızla gidiyor. 2 saatte kaç km yol alır?", "answer": "180", "source": "builtin"},
        {"question": "Ayşe'nin 24 TL'si var. Bir defter 8 TL ise en fazla kaç defter alabilir?", "answer": "3", "source": "builtin"},
        {"question": "5 ile 7'nin toplamının karesi kaçtır?", "answer": "144", "source": "builtin"},
    ]

## HalluVerse-M3 sağlam yükleyici (yeni)

In [ ]:
# ============================================================
# HalluVerse-M3 — SAĞLAM yükleyici (croissant.json baypas + .xlsx desteği)
# ============================================================
import re as _re
def _load_halluverse_m3_robust(repo="sabdalja/HalluVerse-M3"):
    """
    load_dataset() bozuk croissant.json'a takılıyor. Bunun yerine repo dosyalarını
    DOĞRUDAN indirip okur. HalluVerse-M3 verisi .xlsx olarak gelir; Türkçe QA dosyası
    (FINAL_DATASET/QA/QA_tr.xlsx) seçilir. Her satır -> faithful(label=0)+hallu(label=1).
    """
    try:
        from huggingface_hub import HfApi, hf_hub_download
        import pandas as pd
    except Exception as e:
        print(f"  ✗ huggingface_hub/pandas yok: {e}"); return []
    try:
        files = HfApi().list_repo_files(repo, repo_type="dataset")
    except Exception as e:
        print(f"  ✗ repo listelenemedi: {e}"); return []

    DATA_EXT = (".parquet", ".csv", ".tsv", ".json", ".jsonl", ".xlsx", ".xls")
    data_files = [f for f in files
                  if f.lower().endswith(DATA_EXT) and "croissant" not in f.lower()]
    if not data_files:
        print(f"  ✗ veri dosyası yok. Repo: {files[:20]}"); return []

    # Türkçe dosyaları seç; QA'yı (summarization yerine) önceliklendir
    tr_files = [f for f in data_files if _re.search(r'(^|[_/])tr[._]', f.lower())]
    qa_tr = [f for f in tr_files if "qa" in f.lower()]
    chosen = qa_tr or tr_files or data_files
    print(f"  ℹ️  Seçilen dosya(lar): {chosen}")

    def _read(path, fname):
        fl = fname.lower()
        if fl.endswith((".xlsx", ".xls")): return pd.read_excel(path)
        if fl.endswith(".parquet"):        return pd.read_parquet(path)
        if fl.endswith(".tsv"):            return pd.read_csv(path, sep="\t")
        if fl.endswith(".csv"):            return pd.read_csv(path)
        return pd.read_json(path, lines=fl.endswith(".jsonl"))

    frames = []
    for f in chosen:
        try:
            frames.append(_read(hf_hub_download(repo, f, repo_type="dataset"), f))
        except Exception as e:
            print(f"  ⚠️  {f} okunamadı: {e}")
    if not frames:
        return []
    df = pd.concat(frames, ignore_index=True)
    print(f"  ℹ️  Sütunlar: {list(df.columns)} | {len(df)} satır")

    cols = {c.lower(): c for c in df.columns}
    def pick(*cands):
        for c in cands:
            if c in cols: return cols[c]
        return None
    c_q    = pick("question", "prompt", "input", "context", "query")
    c_corr = pick("answer", "reference", "correct_answer", "faithful",
                  "original", "ground_truth", "target", "correct")
    c_hall = pick("edited_answer", "hallucinated_answer", "hallucinated",
                  "hallucination", "edited", "negative", "incorrect")
    c_type = pick("hallucination_type", "type", "category", "error_type", "label")

    out = []
    for _, r in df.iterrows():
        q = str(r[c_q]).strip() if c_q and pd.notna(r[c_q]) else ""
        if not q:
            continue
        htype = str(r[c_type]).strip().lower() if c_type and pd.notna(r[c_type]) else "unknown"
        if c_corr and pd.notna(r[c_corr]) and str(r[c_corr]).strip():
            out.append({"question": q, "expected": str(r[c_corr]).strip(),
                        "type": "factual", "label": 0, "hall_type": htype,
                        "task": "qa", "source": repo})
        if c_hall and pd.notna(r[c_hall]) and str(r[c_hall]).strip():
            out.append({"question": q, "expected": str(r[c_hall]).strip(),
                        "type": htype, "label": 1, "hall_type": htype,
                        "task": "qa", "source": repo})

    n1 = sum(1 for x in out if x["label"] == 1)
    n0 = sum(1 for x in out if x["label"] == 0)
    print(f"  ✓ HalluVerse-M3 (robust): {len(out)} örnek ({n1} hallu / {n0} faithful)")
    return out


def prepare_hallucination_v2():
    """Önce sağlam HF yükleyici; olmazsa orijinal (QA_tr.xlsx / builtin) yola düş."""
    print("\n📚 Halüsinasyon verisi (v2) yükleniyor...")
    qs = _load_halluverse_m3_robust()
    if qs:
        random.seed(CONFIG["seed"]); random.shuffle(qs)
        qs = qs[:CONFIG["max_samples_per_benchmark"]]
        print(f"  → HalluVerse-M3: {len(qs)} örnek hazır")
        return qs
    print("  ⚠️  Robust HF başarısız — orijinal yükleyiciye düşülüyor")
    return prepare_hallucination_custom()


## Model yükleme (orijinalden korundu)

In [ ]:
# ============================================================
# 4. MODEL YÜKLEYİCİ
# ============================================================

def load_model(model_config):
    """Model ve tokenizer yükle (quantization ile)"""
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    model_id = model_config["hf_id"]
    print(f"\n{'='*60}")
    print(f"📦 Model yükleniyor: {model_config['name']}")
    print(f"   HF ID: {model_id}")
    print(f"{'='*60}")

    start = time.time()

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto",
    }

    if CONFIG["quantization"] == "4bit":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        load_kwargs["quantization_config"] = bnb_config
        print("  → 4-bit quantization aktif (BitsAndBytes NF4)")
    elif CONFIG["quantization"] == "8bit":
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        load_kwargs["quantization_config"] = bnb_config
        print("  → 8-bit quantization aktif")
    else:
        load_kwargs["torch_dtype"] = torch.float16
        print("  → FP16 (quantization yok)")

    model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    model.eval()

    elapsed = time.time() - start
    print(f"  ✓ Model yüklendi ({elapsed:.1f}s)")

    return model, tokenizer


def unload_model(model, tokenizer):
    """Model ve tokenizer'ı bellekten sil"""
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("  🗑️ Model bellekten silindi")

## Düzeltilmiş değerlendirme çekirdeği (v2)

In [ ]:
# Düzeltilmiş değerlendirme çekirdeği (greedy + log-likelihood + TruthfulQA-MC)
import math, re, torch
# NOT: CONFIG_V2 yapılandırma hücresinde tanımlı.

# ======================================================================
# 1. ÇEKIRDEK: bir devamın (continuation) koşullu log-olasılığı
# ======================================================================
def _build_context(tokenizer, prompt: str, model_type: str) -> str:
    """Instruct modelde native chat template, base modelde ham prompt."""
    if model_type == "instruct" and getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
        )
    return prompt


@torch.no_grad()
def _continuation_logprob(model, tokenizer, context: str, continuation: str,
                          model_type: str = "instruct"):
    """
    log P(continuation | context) ve continuation token sayısını döndürür.

    Bağlam ve devam AYRI tokenize edilip birleştirilir; böylece devam token'larının
    konumu kesin bilinir (tokenizasyon sınır etkisi yaşanmaz). Üretim/parsing YOK.
    """
    ctx_text = _build_context(tokenizer, context, model_type)
    # Chat template zaten BOS ekliyorsa tekrar eklememek için:
    add_special = not (model_type == "instruct" and getattr(tokenizer, "chat_template", None))
    ctx_ids = tokenizer(ctx_text, add_special_tokens=add_special).input_ids
    cont_ids = tokenizer(continuation, add_special_tokens=False).input_ids
    if len(cont_ids) == 0:
        return -1e9, 0

    input_ids = torch.tensor([ctx_ids + cont_ids], device=model.device)
    # Çok uzun bağlamı baştan kırp (devamı koru)
    if input_ids.shape[1] > CONFIG_V2["max_ctx_len"]:
        overflow = input_ids.shape[1] - CONFIG_V2["max_ctx_len"]
        input_ids = input_ids[:, overflow:]
        ctx_len = len(ctx_ids) - overflow
    else:
        ctx_len = len(ctx_ids)

    logits = model(input_ids).logits  # [1, T, V]
    log_probs = torch.log_softmax(logits.float(), dim=-1)

    total = 0.0
    # i. konumdaki logit, (i+1). token'ı tahmin eder.
    for j, tok_id in enumerate(cont_ids):
        pos = ctx_len + j - 1
        if pos < 0:
            continue
        total += log_probs[0, pos, tok_id].item()
    return total, len(cont_ids)


# ======================================================================
# 2. ÇOKTAN SEÇMELI (MMLU / ARC / HellaSwag) — log-likelihood
# ======================================================================
_LETTERS = ["A", "B", "C", "D", "E"]


def _normalize_gold(answer, n_choices):
    """answer alanını harf indeksine (0..n-1) çevir. int / 'A' / '0' destekler."""
    if isinstance(answer, int):
        return answer if 0 <= answer < n_choices else None
    if isinstance(answer, str):
        a = answer.strip().upper()
        if a in _LETTERS[:n_choices]:
            return _LETTERS.index(a)
        if a.isdigit():
            idx = int(a)
            return idx if 0 <= idx < n_choices else None
    return None


def run_mc_loglikelihood(model, tokenizer, questions, task: str = "mmlu",
                         model_type: str = "instruct"):
    """
    Çoktan seçmeli görevleri PARSING'siz skorlar.
      task="mmlu"/"arc": harf-skorlama (devam = " A"/" B"/...). Prompt'ta şıklar listelenir.
      task="hellaswag" : seçenek-metni skorlama (devam = tam ending), uzunluk normalize.
    Döner: accuracy, macro_f1, item_correct (McNemar için 0/1 listesi), details.
    """
    is_hellaswag = (task == "hellaswag")
    correct, total = 0, 0
    item_correct = []          # McNemar için
    conf_counts = {}           # macro-F1 için sınıf bazlı sayım
    details = []

    for q in questions:
        if is_hellaswag:
            context = q["context"]
            options = q["endings"]
            gold = q.get("label")
            gold = gold if isinstance(gold, int) and 0 <= gold < len(options) else None
            base_prompt = context  # HellaSwag: bağlam + devam; ayrı talimat yok
        else:
            options = q["choices"][:4]
            n = len(options)
            choice_block = "\n".join(f"{_LETTERS[i]}. {c}" for i, c in enumerate(options))
            if model_type == "instruct":
                base_prompt = (f"Aşağıdaki çoktan seçmeli soruyu yanıtla. "
                               f"Sadece doğru seçeneğin harfini (A, B, C veya D) yaz.\n\n"
                               f"Soru: {q['question']}\n{choice_block}\n\nCevap:")
            else:
                base_prompt = f"Soru: {q['question']}\n{choice_block}\n\nDoğru cevap:"
            gold = _normalize_gold(q.get("answer", q.get("answer_key", "")), n)

        if len(options) < 2 or gold is None:
            continue

        # Her seçenek için log-olasılık
        scores = []
        for i, opt in enumerate(options):
            if is_hellaswag:
                cont = " " + str(opt).strip()
            else:
                cont = " " + _LETTERS[i]    # harf-skorlama
            lp, ntok = _continuation_logprob(model, tokenizer, base_prompt, cont, model_type)
            # HellaSwag'da uzunluk normalize (uzun ending cezalanmasın)
            scores.append(lp / max(ntok, 1) if is_hellaswag else lp)

        pred = int(max(range(len(scores)), key=lambda i: scores[i]))
        is_ok = (pred == gold)
        correct += int(is_ok)
        total += 1
        item_correct.append(int(is_ok))

        c = conf_counts.setdefault(gold, {"tp": 0, "fp": 0, "fn": 0})
        if pred == gold:
            c["tp"] += 1
        else:
            conf_counts.setdefault(pred, {"tp": 0, "fp": 0, "fn": 0})["fp"] += 1
            c["fn"] += 1

        details.append({"question": q.get("question", q.get("context", ""))[:160],
                        "pred": pred, "gold": gold, "correct": is_ok})

    accuracy = correct / total if total else 0.0
    f1s = []
    for c in conf_counts.values():
        pr = c["tp"] / (c["tp"] + c["fp"]) if (c["tp"] + c["fp"]) else 0.0
        rc = c["tp"] / (c["tp"] + c["fn"]) if (c["tp"] + c["fn"]) else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if (pr + rc) else 0.0)
    macro_f1 = sum(f1s) / len(f1s) if f1s else 0.0

    print(f"  ✓ {task.upper()} (log-likelihood): {correct}/{total} = "
          f"Acc {accuracy*100:.1f}% | Macro-F1 {macro_f1*100:.1f}%")
    return {"accuracy": accuracy, "macro_f1": macro_f1, "correct": correct,
            "total": total, "item_correct": item_correct, "details": details}


# ======================================================================
# 3. TruthfulQA-MC (MC1 / MC2) — ROUGE-L'in yerine
# ======================================================================
def run_truthfulqa_mc(model, tokenizer, questions, model_type: str = "instruct"):
    """
    Standart TruthfulQA çoktan seçmeli skoru. correct_answers / incorrect_answers
    alanlarını kullanır (generation config bunları zaten sağlar).
      MC1: en yüksek olasılıklı tek aday DOĞRU mu? (0/1)
      MC2: doğru adaylara düşen normalize olasılık kütlesi (0-1).
    Üretim YOK, ROUGE-L YOK, anlamı tersine çeviren kelime tuzağı YOK.
    """
    mc1_list, mc2_list, details = [], [], []

    for q in questions:
        corr = [a for a in q.get("correct_answers", []) if a]
        inc = [a for a in q.get("incorrect_answers", []) if a]
        if q.get("best_answer") and q["best_answer"] not in corr:
            corr = [q["best_answer"]] + corr
        if not corr or not inc:
            continue   # MC için her iki taraf da gerekli

        prompt = f"Soru: {q['question']}\nCevap:"
        cands = [(ans, True) for ans in corr] + [(ans, False) for ans in inc]
        norm_lps = []
        for ans, _ in cands:
            lp, ntok = _continuation_logprob(model, tokenizer, prompt, " " + ans.strip(),
                                             model_type)
            norm_lps.append(lp / max(ntok, 1))   # uzunluk normalize

        # MC1: argmax doğru tarafta mı
        best = int(max(range(len(cands)), key=lambda i: norm_lps[i]))
        mc1 = 1 if cands[best][1] else 0

        # MC2: softmax olasılık kütlesi (doğrular / toplam)
        m = max(norm_lps)
        exps = [math.exp(lp - m) for lp in norm_lps]
        Z = sum(exps)
        mc2 = sum(e for e, (_, t) in zip(exps, cands) if t) / Z if Z > 0 else 0.0

        mc1_list.append(mc1)
        mc2_list.append(mc2)
        details.append({"question": q["question"][:160], "mc1": mc1, "mc2": round(mc2, 4)})

    mc1_acc = sum(mc1_list) / len(mc1_list) if mc1_list else 0.0
    mc2_avg = sum(mc2_list) / len(mc2_list) if mc2_list else 0.0
    print(f"  ✓ TruthfulQA-MC: MC1 {mc1_acc*100:.1f}% | MC2 {mc2_avg*100:.1f}% "
          f"(n={len(mc1_list)})")
    return {"mc1": mc1_acc, "mc2": mc2_avg, "total": len(mc1_list),
            "item_correct": mc1_list, "mc2_per_item": mc2_list, "details": details}


# ======================================================================
# 4. GSM8K — greedy üretim (matematik üretken görev; MC değil)
# ======================================================================
@torch.no_grad()
def _generate_greedy(model, tokenizer, prompt, model_type, max_tokens):
    text = _build_context(tokenizer, prompt, model_type)
    add_special = not (model_type == "instruct" and getattr(tokenizer, "chat_template", None))
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=CONFIG_V2["max_ctx_len"],
                       add_special_tokens=add_special).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    new = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()


def _extract_number(text):
    """Son sayıyı çıkar. ESKI BUG düzeltildi: ondalık nokta artık korunuyor."""
    if text is None:
        return None
    s = str(text)
    m = re.search(r'####\s*(-?\d[\d.,]*)', s)
    cand = m.group(1) if m else None
    if cand is None:
        nums = re.findall(r'-?\d[\d.,]*', s)
        cand = nums[-1] if nums else None
    if cand is None:
        return None
    cand = cand.replace(",", "")          # binlik ayıracı at, ondalık noktayı KORU
    try:
        return float(cand)
    except ValueError:
        return None


# ----------------------------------------------------------------------
# GSM8K few-shot (YALNIZCA base modeller icin — Hakem #4)
# ----------------------------------------------------------------------
# En profesyonel yol: ornekleri GSM8K-TR'nin TRAIN bolumunden cekmektir (test'ten
# DEGIL -> sizinti yok). Asagidaki yardimci, train split'i erisilebilirse 3 gercek
# ornegi GSM8K_FEWSHOT degiskenine yazar. Erisilemezse, asagidaki elle yazilmis,
# TEST setinde bulunmayan ve ayni cok-adimli GSM8K-TR tarziyla uyumlu varsayilan
# ornekler kullanilir. Her iki durumda da degerlendirilen 500 test sorusu ornek
# olarak KULLANILMAZ.

_GSM8K_FEWSHOT_DEFAULT = (
    "Problem: Bir kitapci, fiyati 40 lira olan bir kitapta %25 indirim yapiyor. "
    "Bir musteri bu kitaptan 3 adet satin aliyor. Musteri toplam kac lira oder?\n"
    "Çözüm: Indirim tutari 40 * 0.25 = 10 lira oldugundan indirimli fiyat 40 - 10 = 30 "
    "liradir. Musteri 3 kitap aldigi icin toplam 3 * 30 = 90 lira oder. #### 90\n\n"
    "Problem: Bir depoda 120 kg un vardir. Ilk gun bunun dortte biri, ikinci gun ise "
    "kalan unun yarisi kullaniliyor. Geriye kac kg un kalir?\n"
    "Çözüm: Ilk gun 120 * (1/4) = 30 kg un kullanilir ve geriye 120 - 30 = 90 kg kalir. "
    "Ikinci gun kalanin yarisi, yani 90 / 2 = 45 kg kullanilir. Sonunda 90 - 45 = 45 kg "
    "un kalir. #### 45\n\n"
    "Problem: Ayse once saatte 12 km hizla 2 saat, ardindan saatte 8 km hizla 3 saat "
    "yuruyor. Ayse toplam kac km yol gitmistir?\n"
    "Çözüm: Ilk bolumde 12 * 2 = 24 km, ikinci bolumde 8 * 3 = 24 km gider. Toplam "
    "yol 24 + 24 = 48 km olur. #### 48\n\n"
)

def build_gsm8k_fewshot_from_train(k=3):
    """GSM8K-TR train bolumunden k adet ornekle few-shot blogu uretir (opsiyonel).
    Basarisiz olursa None doner; bu durumda _GSM8K_FEWSHOT_DEFAULT kullanilir."""
    try:
        import re
        from datasets import load_dataset
        ds = load_dataset("malhajar/gsm8k_tr-v0.2", split="train")
        parts = []
        for i in range(k):
            ex = ds[i]
            ans = re.sub(r"<<.*?>>", "", str(ex["answer"])).strip()
            parts.append(f"Problem: {ex['question']}\nÇözüm: {ans}")
        return "\n\n".join(parts) + "\n\n"
    except Exception as e:
        print("  ℹ️ train few-shot cekilemedi, varsayilan ornekler kullanilacak:", str(e)[:80])
        return None

# Train'den cekmeyi DENE (istege bagli). Basarisizsa varsayilan devreye girer.
GSM8K_FEWSHOT = build_gsm8k_fewshot_from_train(3) or _GSM8K_FEWSHOT_DEFAULT


def run_gsm8k_greedy(model, tokenizer, questions, model_type="instruct"):
    correct, total, item_correct, details = 0, 0, [], []
    for q in questions:
        if model_type == "instruct":
            prompt = (f"Aşağıdaki matematik problemini adım adım çöz.\n"
                      f"Sonucu en sona '#### <sayı>' formatında yaz.\n\n"
                      f"Problem: {q['question']}\n\nÇözüm:")
        else:
            # BASE model (Kumru-2B): talimat takip edemez. Format uyumu icin few-shot
            # verilir (Hakem #4). Oncelik train split'inden uretilen GSM8K_FEWSHOT;
            # yoksa TEST setinde BULUNMAYAN, GSM8K-TR tarziyla uyumlu varsayilan ornekler.
            _fewshot = GSM8K_FEWSHOT if "GSM8K_FEWSHOT" in globals() else _GSM8K_FEWSHOT_DEFAULT
            prompt = _fewshot + f"Problem: {q['question']}\nÇözüm:"
        resp = _generate_greedy(model, tokenizer, prompt, model_type, max_tokens=300)
        pred, gold = _extract_number(resp), _extract_number(str(q["answer"]))
        ok = pred is not None and gold is not None and abs(pred - gold) < 1e-2
        correct += int(ok); total += 1; item_correct.append(int(ok))
        details.append({"pred": pred, "gold": gold, "correct": ok,
                        "raw": resp[:200]})
    em = correct / total if total else 0.0
    print(f"  ✓ GSM8K-TR (greedy): {correct}/{total} = EM {em*100:.1f}%")
    return {"exact_match": em, "correct": correct, "total": total,
            "item_correct": item_correct, "details": details}


# ======================================================================
# 5. HalluVerse — sınıf dağılımı + ikili metrikler (greedy)
# ======================================================================
def run_hallucination(model, tokenizer, questions, model_type="instruct"):
    """
    HalluVerse-M3 ikili sınıflandırma. Eski mantık korunur ama:
      - greedy üretim,
      - SINIF DAĞILIMI raporlanır (Hakem #2),
      - F1 yanında pozitif/negatif oranı ve dengesizlik açıkça gösterilir,
      - item_correct McNemar için döndürülür.
    """
    rows = []
    n_pos = sum(1 for q in questions if q.get("label") == 1)
    n_neg = sum(1 for q in questions if q.get("label") == 0)

    for q in questions:
        cand = q.get("expected", "")
        if model_type == "instruct":
            prompt = (f"Aşağıdaki soru ve cevap çiftini değerlendir. Cevap doğru ve "
                      f"gerçekçi ise 'DOĞRU', yanlış/uydurma ise 'YANLIŞ' yaz. "
                      f"Tek kelime.\n\nSoru: {q['question']}\nCevap: {cand}\n\nDeğerlendirme:")
        else:
            prompt = f"Soru: {q['question']}\nCevap: {cand}\nBu cevap doğru mu? (DOĞRU/YANLIŞ):"
        resp = _generate_greedy(model, tokenizer, prompt, model_type, max_tokens=8).upper()
        says_wrong = any(t in resp for t in ("YANLIŞ", "YANLIS", "FALSE", "HATALI"))
        says_right = any(t in resp for t in ("DOĞRU", "DOGRU", "TRUE"))
        # pozitif sınıf = "halüsinasyon var (label=1)"; model YANLIŞ derse pozitif tahmin
        pred = 1 if (says_wrong and not says_right) else 0
        rows.append({"label": q.get("label"), "pred": pred})

    labeled = [r for r in rows if r["label"] is not None]
    tp = sum(1 for r in labeled if r["label"] == 1 and r["pred"] == 1)
    tn = sum(1 for r in labeled if r["label"] == 0 and r["pred"] == 0)
    fp = sum(1 for r in labeled if r["label"] == 0 and r["pred"] == 1)
    fn = sum(1 for r in labeled if r["label"] == 1 and r["pred"] == 0)
    acc = (tp + tn) / len(labeled) if labeled else 0.0
    pr = tp / (tp + fp) if (tp + fp) else 0.0
    rc = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * pr * rc / (pr + rc) if (pr + rc) else 0.0
    item_correct = [int(r["pred"] == r["label"]) for r in labeled]

    print(f"  ✓ HalluVerse-M3: Acc {acc*100:.1f}% | F1 {f1*100:.1f}% "
          f"| P {pr*100:.1f}% R {rc*100:.1f}%")
    print(f"     SINIF DAĞILIMI: pozitif(halüsinasyon)={n_pos}, negatif(faithful)={n_neg} "
          f"(dengesizlik bu yüzden Acc ile F1 birlikte raporlanmalı)")
    return {"accuracy": acc, "f1": f1, "precision": pr, "recall": rc,
            "n_pos": n_pos, "n_neg": n_neg, "tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "item_correct": item_correct, "total": len(labeled)}

## İstatistiksel anlamlılık

In [ ]:
# -*- coding: utf-8 -*-
"""
stats_utils.py — Türkçe LLM benchmark çalışması için istatistiksel anlamlılık araçları.

Hakem #2 ve #4'ün ana itirazını karşılar:
  - "n=100'de Wilson GA yaklaşık ±%5" iddiası YANLIŞ. p=0.5'te gerçek değer ±%9.8.
  - Modeller arası küçük farklar anlamlılık testiyle raporlanmalı (McNemar / bootstrap).

Bu modül yalnızca numpy + scipy gerektirir; GPU/model gerektirmez.
Sonuç JSON'larını okuyup tablo/şekil üretmeden ÖNCE buradan geçirin.
"""
from __future__ import annotations
import math
from itertools import combinations

import numpy as np
from scipy import stats


# ----------------------------------------------------------------------
# 1. Tek bir oran (accuracy / F1-as-rate) için güven aralığı
# ----------------------------------------------------------------------
def wilson_ci(k: int, n: int, conf: float = 0.95):
    """
    Wilson skor güven aralığı (binom oranı için).
    k: doğru sayısı, n: toplam.
    Döner: (p_hat, low, high, half_width) — hepsi 0-1 ölçeğinde.

    NEDEN: Makaledeki '±%5' aslında standart hatadır (sqrt(p(1-p)/n)=0.05).
    Asıl güven aralığı yarı-genişliği p=0.5, n=100 için ~0.098'dir.
    """
    if n == 0:
        return (0.0, 0.0, 0.0, 0.0)
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z / denom) * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    low, high = center - margin, center + margin
    return (p, max(0.0, low), min(1.0, high), (high - low) / 2)


def required_n_for_margin(margin: float = 0.05, conf: float = 0.95, p: float = 0.5):
    """Belirli bir hata payı için gereken minimum örneklem (en kötü durum p=0.5)."""
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    return math.ceil((z**2 * p * (1 - p)) / margin**2)


# ----------------------------------------------------------------------
# 2. Açık uçlu / sürekli metrikler için bootstrap GA (ör. GSM8K EM, MC2)
# ----------------------------------------------------------------------
def bootstrap_ci(per_item_scores, conf: float = 0.95, n_boot: int = 10000, seed: int = 42):
    """
    Örnek-bazlı skor listesi (0/1 ya da sürekli) için bootstrap ortalama GA'sı.
    per_item_scores: her soru için skor (liste/np.array).
    Döner: (mean, low, high).
    """
    rng = np.random.default_rng(seed)
    x = np.asarray(per_item_scores, dtype=float)
    n = len(x)
    if n == 0:
        return (0.0, 0.0, 0.0)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = x[idx].mean(axis=1)
    alpha = (1 - conf) / 2
    return (float(x.mean()),
            float(np.quantile(boot_means, alpha)),
            float(np.quantile(boot_means, 1 - alpha)))


# ----------------------------------------------------------------------
# 3. İki modelin AYNI sorulardaki doğru/yanlışını karşılaştırma: McNemar
# ----------------------------------------------------------------------
def mcnemar_test(correct_a, correct_b, exact_threshold: int = 25):
    """
    Eşleştirilmiş (paired) McNemar testi.
    correct_a, correct_b: aynı soru sırasıyla 0/1 doğruluk listeleri (eşit uzunlukta).
    b = A doğru & B yanlış, c = A yanlış & B doğru.
    n = b + c küçükse (<=25) exact binom; değilse süreklilik düzeltmeli ki-kare.
    Döner: dict(b, c, statistic, p_value, significant_05).
    """
    a = np.asarray(correct_a, dtype=int)
    bb = np.asarray(correct_b, dtype=int)
    assert len(a) == len(bb), "İki model aynı soru kümesinde değerlendirilmeli!"
    b = int(np.sum((a == 1) & (bb == 0)))
    c = int(np.sum((a == 0) & (bb == 1)))
    n = b + c
    if n == 0:
        return {"b": b, "c": c, "statistic": 0.0, "p_value": 1.0, "significant_05": False}
    if n <= exact_threshold:
        # Exact: B(min(b,c); n, 0.5) iki yanlı
        k = min(b, c)
        p = 2 * stats.binom.cdf(k, n, 0.5)
        p = min(1.0, p)
        stat = float(k)
    else:
        stat = (abs(b - c) - 1) ** 2 / n  # Edwards süreklilik düzeltmesi
        p = float(stats.chi2.sf(stat, df=1))
    return {"b": b, "c": c, "statistic": float(stat), "p_value": float(p),
            "significant_05": p < 0.05}


def pairwise_significance(model_correct: dict, metric_name: str = "accuracy"):
    """
    {model_adi: [0/1 doğruluk listesi]} sözlüğü için tüm ikili McNemar p-değerleri.
    Döner: rapor edilebilir satır listesi.
    """
    rows = []
    for m1, m2 in combinations(model_correct.keys(), 2):
        res = mcnemar_test(model_correct[m1], model_correct[m2])
        acc1 = np.mean(model_correct[m1])
        acc2 = np.mean(model_correct[m2])
        rows.append({
            "metric": metric_name,
            "model_a": m1, "model_b": m2,
            "acc_a": round(float(acc1), 4), "acc_b": round(float(acc2), 4),
            "diff": round(float(acc1 - acc2), 4),
            "p_value": round(res["p_value"], 4),
            "significant_05": res["significant_05"],
            "verdict": ("anlamlı fark" if res["significant_05"]
                        else "anlamlı fark YOK (GA örtüşüyor)"),
        })
    return rows


def format_score_with_ci(k: int, n: int, conf: float = 0.95) -> str:
    """Tabloya yazılacak '49.9 [40.0, 59.8]' biçimli metin (yüzde)."""
    p, low, high, hw = wilson_ci(k, n, conf)
    return f"{p*100:.1f} [{low*100:.1f}, {high*100:.1f}]"

## Orkestrasyon & raporlama

In [ ]:
# ============================================================
# ORKESTRASYON — TEK TEK (MANUEL) ÇALIŞTIRMA
# ============================================================
import json, os, glob

def prepare_all_benchmarks():
    """Benchmark'lar BİR KEZ hazırlanır; tüm modeller AYNI soru sırasını görür (McNemar)."""
    return {
        "mmlu":          prepare_mmlu_tr(),
        "arc":           prepare_arc_tr(),
        "hellaswag":     prepare_hellaswag_tr(),
        "gsm8k":         prepare_gsm8k_tr(),
        "truthfulqa":    prepare_truthfulqa_tr(),
        "hallucination": prepare_hallucination_v2(),   # croissant-bypass
    }

def run_one_model(model_key, benches):
    cfg = MODELS[model_key]; mt = cfg["type"]
    model, tok = load_model(cfg)
    out = {}
    try:
        out["mmlu"]          = run_mc_loglikelihood(model, tok, benches["mmlu"], task="mmlu", model_type=mt)
        out["arc"]           = run_mc_loglikelihood(model, tok, benches["arc"], task="arc", model_type=mt)
        out["hellaswag"]     = run_mc_loglikelihood(model, tok, benches["hellaswag"], task="hellaswag", model_type=mt)
        out["gsm8k"]         = run_gsm8k_greedy(model, tok, benches["gsm8k"], model_type=mt)
        out["truthfulqa"]    = run_truthfulqa_mc(model, tok, benches["truthfulqa"], model_type=mt)
        out["hallucination"] = run_hallucination(model, tok, benches["hallucination"], model_type=mt)
    finally:
        unload_model(model, tok)
    return out

# --- Benchmark'ları BİR KEZ hazırla ve diske önbelleğe al ---------------------
# Kernel yeniden başlasa / oturum kapansa bile tüm modeller AYNI soruları görür;
# McNemar eşleştirmesinin geçerli kalması için bu şarttır.
def get_benchmarks(out_dir=CONFIG["output_dir"], force=False):
    os.makedirs(out_dir, exist_ok=True)
    cache = os.path.join(out_dir, "benchmarks.json")
    if (not force) and os.path.exists(cache):
        with open(cache, encoding="utf-8") as f:
            benches = json.load(f)
        print("ℹ️  Önbellekten yüklendi: " + cache + "  ("
              + ", ".join(f"{k}={len(v)}" for k, v in benches.items()) + ")")
        return benches
    benches = prepare_all_benchmarks()
    with open(cache, "w", encoding="utf-8") as f:
        json.dump(benches, f, ensure_ascii=False)
    print("✓ Hazırlandı ve önbelleğe alındı: " + cache + "  ("
          + ", ".join(f"{k}={len(v)}" for k, v in benches.items()) + ")")
    return benches

# --- TEK senaryo (bir model) çalıştır + ANINDA diske kaydet -------------------
def run_and_save(model_key, benches=None, out_dir=CONFIG["output_dir"], keep_details=True):
    assert model_key in MODELS, f"Bilinmeyen model: {model_key}\nSeçenekler: {list(MODELS)}"
    if benches is None:
        benches = get_benchmarks(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    print("\n" + "=" * 60 + f"\n>>> SENARYO: {MODELS[model_key]['name']}\n" + "=" * 60)
    res = run_one_model(model_key, benches)
    # item_correct + mc2_per_item KORUNUR (McNemar/bootstrap için gerekli); 'details' opsiyonel
    slim = {b: {kk: vv for kk, vv in r.items() if keep_details or kk != "details"}
            for b, r in res.items()}
    path = os.path.join(out_dir, f"result_{model_key}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(slim, f, ensure_ascii=False, indent=2,
                  default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print(f"\n💾 KAYDEDİLDİ -> {path}")
    return res

# --- Diske kaydedilmiş TÜM senaryoları topla (rapor için) ---------------------
def load_saved_results(out_dir=CONFIG["output_dir"]):
    results = {}
    for path in sorted(glob.glob(os.path.join(out_dir, "result_*.json"))):
        key = os.path.basename(path)[len("result_"):-len(".json")]
        with open(path, encoding="utf-8") as f:
            results[key] = json.load(f)
    print("Yüklenen senaryolar: " + (", ".join(results) if results else "(yok)"))
    return results


In [ ]:
# ============================================================
# RAPORLAMA — Wilson GA + ikili McNemar (cross-benchmark ORTALAMA YOK)
# Eksik/yarım senaryolara dayanıklı: sadece kaydedilmiş modelleri raporlar.
# ============================================================
import json, os
BINOM_BENCHES = ["mmlu", "arc", "hellaswag", "gsm8k", "truthfulqa", "hallucination"]

def _correct_total(rb):
    correct = rb.get("correct")
    if correct is None and rb.get("item_correct") is not None:
        correct = sum(rb["item_correct"])
    total = rb.get("total") or len(rb.get("item_correct", []))
    return int(correct or 0), int(total or 0)

def report(results):
    if not results:
        print("⚠️  Sonuç yok. Önce en az bir senaryo çalıştırın: run_and_save(\"<model>\", BENCHES)")
        return
    for bench in BINOM_BENCHES:
        print(f"\n### {bench.upper()}  (skor [%95 Wilson GA])")
        for k, r in results.items():
            rb = r.get(bench)
            if rb is None:
                print(f"  {MODELS[k]['name']:<28} (bu senaryoda yok)"); continue
            correct, total = _correct_total(rb)
            line = f"  {MODELS[k]['name']:<28} " + format_score_with_ci(correct, total)
            if bench == "truthfulqa" and rb.get("mc2_per_item"):
                mean, lo, hi = bootstrap_ci(rb["mc2_per_item"])
                line += f"   | MC2 {mean*100:.1f} [{lo*100:.1f}, {hi*100:.1f}]"
            if bench == "hallucination":
                line += f"   | F1 {rb['f1']*100:.1f}  (poz={rb['n_pos']}, neg={rb['n_neg']})"
            print(line)
    for bench in BINOM_BENCHES:
        item = {k: results[k][bench]["item_correct"] for k in results
                if results[k].get(bench, {}).get("item_correct") is not None}
        if len(item) < 2:
            continue   # ikili karşılaştırma için en az 2 senaryo gerekir
        rows = pairwise_significance(item, bench)
        sig = [x for x in rows if x["significant_05"]]
        print(f"\n### {bench.upper()} — anlamlı (p<0.05) ikili fark: {len(sig)}/{len(rows)}")
        for x in sig:
            print(f"   {x['model_a']} vs {x['model_b']}: delta={x['diff']*100:+.1f} p={x['p_value']}")
    print("\nNot: Cross-benchmark tek 'ortalama puan' RAPORLANMAZ (farklı metrik/ölçek).")

def save_all(results, out_dir=CONFIG["output_dir"]):
    os.makedirs(out_dir, exist_ok=True)
    slim = {k: {b: {kk: vv for kk, vv in r.items() if kk != "details"}
                for b, r in res.items()} for k, res in results.items()}
    with open(f"{out_dir}/results.json", "w", encoding="utf-8") as f:
        json.dump(slim, f, ensure_ascii=False, indent=2,
                  default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print(f"OK -> {out_dir}/results.json")


## Çalıştır — senaryoları TEK TEK (manuel)

Otomatik döngü (`run_all_v2`) kaldırıldı. Akış:

1. **Adım 0**'ı bir kez çalıştır → benchmark'lar hazırlanıp `asyu_results/benchmarks.json`'a önbelleklenir. Böylece tüm modeller **aynı** soruları görür (McNemar geçerliliği). İlk hazırlıktan sonra bu hücre önbellekten okur.
2. **Adım 1**'de istediğin modelin hücresini çalıştır. Her hücre: modeli yükler → 6 benchmark → `asyu_results/result_<model>.json`'a **anında kaydeder** → modeli bellekten siler. İstediğin sırada, istediğin kadarını koşabilirsin; kernel yeniden başlasa bile kayıtlar diskte kalır.
3. Hepsi bitince **Adım 2**'yi çalıştır → diskteki tüm `result_*.json` dosyalarını toplayıp raporlar.

> Bir senaryoyu yeniden koşmak: ilgili hücreyi tekrar çalıştır (dosyanın üzerine yazılır). Soruları sıfırdan hazırlamak için: `get_benchmarks(force=True)`.

In [ ]:
# === Adım 0-ÖNCESİ: Benchmark'ları kırp (süreyi kısaltır, hepsi ≥384 → Hakem #4 eşiği) ===
# ÖNEMLİ: Çalışan run'ı DURDUR, bu hücreyi çalıştır, SONRA Adım 0'ı çalıştırıp
# 7 modeli de baştan koş. Böylece tüm modeller AYNI kırpılmış soruları görür (McNemar geçerli).
# Yeniden indirme YOK; mevcut asyu_results/benchmarks.json yerinde kırpılır.
import json, os

cache = os.path.join(CONFIG["output_dir"], "benchmarks.json")
assert os.path.exists(cache), "Önce benchmarks.json üretilmeli (get_benchmarks bir kez koşmuş olmalı)."
benches = json.load(open(cache, encoding="utf-8"))

# Sıra seed=42 ile karışık olduğundan ilk N'i almak = deterministik rastgele alt küme.
TARGETS = {
    "mmlu":          2905,   # TAM tut (MC hızlı, ~1 sn/soru; manşet sonuç + güçlü yön)
    "arc":           1169,   # tam
    "hellaswag":     1000,   # 10041 -> 1000  (en büyük MC zaman kazancı)
    "gsm8k":          500,   # 1317  -> 500   (üretken; asıl darboğaz)
    "truthfulqa":     500,   # 816   -> 500
    "hallucination":  800,   # 1560  -> 800
}

for k, n in TARGETS.items():
    before = len(benches[k])
    benches[k] = benches[k][:n]
    assert len(benches[k]) >= 384, f"{k}: {len(benches[k])} < 384 (Hakem #4 eşiği altında!)"
    print(f"  {k:14} {before:5} -> {len(benches[k]):5}")

json.dump(benches, open(cache, "w", encoding="utf-8"), ensure_ascii=False)
print("✓ Kırpıldı ve kaydedildi ->", cache)
print("Şimdi Adım 0'ı (BENCHES = get_benchmarks()) çalıştır; kırpılmış dosyayı okuyacak.")

In [ ]:
# === Drive'a kalıcı kayıt kurulumu (model hücrelerinden ÖNCE, bir kez) ===
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/asyu_results"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Kırpılmış benchmarks.json'ı Drive'a taşı (yoksa get_benchmarks yeniden üretir → set değişir!)
src = os.path.join(CONFIG["output_dir"], "benchmarks.json")
dst = os.path.join(DRIVE_DIR, "benchmarks.json")
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy(src, dst)
    print("✓ benchmarks.json Drive'a kopyalandı")

# Bundan sonra TÜM kayıtlar buraya gider
CONFIG["output_dir"] = DRIVE_DIR
print("✓ Çıktı dizini ->", CONFIG["output_dir"])

In [ ]:
BENCHES = get_benchmarks(out_dir=CONFIG["output_dir"])

In [ ]:
# === Adım 0: Benchmark'ları bir kez hazırla (önbelleğe alınır) ===
BENCHES = get_benchmarks()

In [ ]:
import inspect
print("run_and_save default :", inspect.signature(run_and_save).parameters["out_dir"].default)
print("get_benchmarks default:", inspect.signature(get_benchmarks).parameters["out_dir"].default)
print("CONFIG                :", CONFIG["output_dir"])

In [ ]:
# === VRAM'i temizle (model hücreleri ARASINDA, gerekirse) ===
import gc, torch

# Eğer ortamda yüklü model/tokenizer değişkeni kaldıysa önce onları sil
for _v in ["model", "tok", "tokenizer"]:
    if _v in globals():
        del globals()[_v]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Durumu göster
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    used = (total - free) / 1024**3
    print(f"VRAM kullanımda: {used:.2f} GB / {total/1024**3:.2f} GB")
    print(torch.cuda.memory_summary(abbreviated=True))
else:
    print("CUDA yok (CPU ortamı)")

In [ ]:
import gc, torch

# Tüm CUDA tensörlerini bul ve referansları kopar
for obj in list(globals().keys()):
    try:
        if torch.is_tensor(globals()[obj]) or hasattr(globals()[obj], "is_cuda"):
            del globals()[obj]
    except Exception:
        pass

# Olası model/tokenizer isimlerini de temizle
for _v in ["model", "tok", "tokenizer", "m", "t"]:
    globals().pop(_v, None)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

free, total = torch.cuda.mem_get_info()
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved : {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print(f"Boşta    : {free/1024**3:.2f} GB / {total/1024**3:.2f} GB")

In [ ]:
import gc, torch

# CUDA'da yer kaplayan tüm modülleri/tensörleri garbage collector üzerinden bul
killed = 0
for obj in gc.get_objects():
    try:
        # nn.Module (model) veya CUDA tensörü mü?
        if isinstance(obj, torch.nn.Module):
            obj.to("cpu")
            killed += 1
        elif torch.is_tensor(obj) and obj.is_cuda:
            obj.data = obj.data.cpu()
            killed += 1
    except Exception:
        pass

print(f"CPU'ya taşınan nesne: {killed}")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved : {torch.cuda.memory_reserved()/1024**3:.2f} GB")

### Adım 1 — Senaryoları tek tek çalıştır
Her hücre tek bir modeli koşar ve sonucunu diske yazar. İstediğini çalıştır.

In [ ]:
# Senaryo: cosmos_gemma_9b
run_and_save("cosmos_gemma_9b", BENCHES, out_dir=CONFIG["output_dir"])

In [ ]:
# Senaryo: trendyol_7b_chat
run_and_save("trendyol_7b_chat", BENCHES, out_dir=CONFIG["output_dir"])

In [ ]:
# Senaryo: trendyol_7b_chat_v4
run_and_save("trendyol_7b_chat_v4", BENCHES, out_dir=CONFIG["output_dir"])

In [ ]:
# Senaryo: wiroai_9b
run_and_save("wiroai_9b", BENCHES, out_dir=CONFIG["output_dir"])

In [ ]:
# Senaryo: wiroai_8b
run_and_save("wiroai_8b", BENCHES, out_dir=CONFIG["output_dir"])

In [ ]:
# Senaryo: kocdigital_8b
run_and_save("kocdigital_8b", BENCHES, out_dir=CONFIG["output_dir"])

In [ ]:
# Few-shot içeriğini ve kaynağını gör
print("GSM8K_FEWSHOT tanımlı mı:", "GSM8K_FEWSHOT" in globals())
print("="*60)
print(GSM8K_FEWSHOT)
print("="*60)
print("Örnek sayısı (Problem: kaç kez):", GSM8K_FEWSHOT.count("Problem:"))
print("Her örnek #### ile bitiyor mu:", GSM8K_FEWSHOT.count("####"))

In [ ]:
# Kumru'nun GSM8K'da göreceği TAM prompt'u simüle et (model gerekmez)
_fewshot = GSM8K_FEWSHOT if "GSM8K_FEWSHOT" in globals() else _GSM8K_FEWSHOT_DEFAULT
ornek_soru = BENCHES["gsm8k"][0]["question"]
tam_prompt = _fewshot + f"Problem: {ornek_soru}\nÇözüm:"
print(tam_prompt)

In [ ]:
# instruct dalı few-shot ALMAMALI (zero-shot kalmalı)
q = BENCHES["gsm8k"][0]
instruct_prompt = (f"Aşağıdaki matematik problemini adım adım çöz.\n"
                   f"Sonucu en sona '#### <sayı>' formatında yaz.\n\n"
                   f"Problem: {q['question']}\n\nÇözüm:")
print("instruct prompt'ta few-shot örneği VAR mı (olmamalı):", "kitapci" in instruct_prompt or "Frances" in instruct_prompt)
# False çıkmalı

In [ ]:
# Senaryo: kumru_2b
run_and_save("kumru_2b_base", BENCHES, out_dir=CONFIG["output_dir"], keep_details=False)

### Adım 2 — Tüm kayıtlı senaryoları topla ve raporla

In [ ]:
results = load_saved_results(out_dir=CONFIG["output_dir"])
print("Yüklenen modeller:", list(results.keys()))   # 7 olmalı

In [ ]:
report(results)
save_all(results)

In [ ]:
# === Adım 2: Diskteki tüm sonuçları topla → rapor + birleşik kayıt ===
results = load_saved_results()
report(results)
save_all(results)